# 05 CNN Review

This notebook reviews compact CNN artifacts if you chose to train them on GPU. It is optional until the classical baseline pipeline is stable.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/home/ubuntu/nishn_workspce/test_pdfs_generic/.covid_audio_btp_private/covid_audio_btp")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    artifact,
    check_artifacts,
    count_table,
    read_csv,
    read_optional_csv,
    require_artifacts,
    save_table,
    value_counts_frame,
    assert_no_participant_leakage,
    assert_binary_labels_present,
    stop_if_validation_errors,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)


In [ ]:
cnn_metric_files = sorted((PROJECT_ROOT / "data/outputs/metrics").glob("cnn_metrics*.csv"))
cnn_history_files = sorted((PROJECT_ROOT / "data/outputs/metrics").glob("cnn_training_history*.csv"))
cnn_val_files = sorted((PROJECT_ROOT / "data/outputs/metrics").glob("cnn_logits_validation*.csv"))
cnn_test_files = sorted((PROJECT_ROOT / "data/outputs/metrics").glob("cnn_logits_test*.csv"))
print("metric files", [p.name for p in cnn_metric_files])
print("history files", [p.name for p in cnn_history_files])
if not cnn_metric_files:
    raise FileNotFoundError("No CNN metrics found. Keep this notebook for after RUN_CNN=True on GPU.")


## CNN Metrics


In [ ]:
cnn_metrics = pd.concat([pd.read_csv(p).assign(source_file=p.name) for p in cnn_metric_files], ignore_index=True)
metric_cols = [c for c in ["model_name", "modality", "auroc", "auprc", "balanced_accuracy", "f1", "brier", "ece", "n_samples", "source_file"] if c in cnn_metrics.columns]
save_table(cnn_metrics[metric_cols], "reports/tables/nb05_cnn_metrics.csv")
display(cnn_metrics[metric_cols].sort_values("auprc", ascending=False))


## Training Curves


In [ ]:
if cnn_history_files:
    histories = []
    for p in cnn_history_files:
        h = pd.read_csv(p)
        h["source_file"] = p.name
        histories.append(h)
    history = pd.concat(histories, ignore_index=True)
    display(history.groupby("source_file")[["train_loss", "validation_loss"]].tail(3))
    plt.figure(figsize=(9, 5))
    for source, group in history.groupby("source_file"):
        plt.plot(group["epoch"], group["train_loss"], label=f"{source} train")
        plt.plot(group["epoch"], group["validation_loss"], linestyle="--", label=f"{source} val")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "reports/figures/nb05_cnn_training_curves.png", dpi=160)
    plt.show()
else:
    print("No CNN history files found.")


## CNN Probability And Logit Checks


In [ ]:
pred_files = cnn_val_files + cnn_test_files
if pred_files:
    cnn_pred = pd.concat([pd.read_csv(p).assign(source_file=p.name) for p in pred_files], ignore_index=True)
    display(cnn_pred.groupby(["source_file", "modality", "split"])["probability"].describe().reset_index())
    if "logit" in cnn_pred.columns:
        sns.histplot(data=cnn_pred, x="logit", hue="label_binary", bins=50, element="step")
        plt.title("CNN logits by label")
        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / "reports/figures/nb05_cnn_logits.png", dpi=160)
        plt.show()
else:
    print("No CNN prediction files found.")


## Compare CNN With Classical Baselines


In [ ]:
ml = read_optional_csv("data/outputs/metrics/ml_baseline_metrics.csv")
if ml is not None:
    best_ml = ml.sort_values("auprc", ascending=False).groupby("modality").head(1)[["modality", "model_name", "auprc", "auroc", "ece"]]
    best_cnn = cnn_metrics.sort_values("auprc", ascending=False).groupby("modality").head(1)[["modality", "model_name", "auprc", "auroc", "ece"]]
    comparison = best_ml.merge(best_cnn, on="modality", suffixes=("_ml", "_cnn"), how="outer")
    save_table(comparison, "reports/tables/nb05_cnn_vs_ml.csv")
    display(comparison)
